# XPT2046 Resistive Touchscreen Tutorial

This tutorial demonstrates how to use the XPT2046 resistive touchscreen driver with STM32F429I-DISC1.

## Overview

The XPT2046 is a popular resistive touchscreen controller providing:
- 12-bit ADC resolution (4096 x 4096 points)
- Touch pressure detection
- SPI interface communication
- Interrupt-driven touch detection
- Coordinate calibration and mapping

## Hardware Setup

### Required Connections

| XPT2046 Pin | STM32F429 Pin | Function |
|-------------|---------------|----------|
| VCC | 3.3V | Power supply |
| GND | GND | Ground |
| CS | PB15 | Chip select |
| DIN | PF9 | SPI data in (SPI5) |
| DOUT | PF8 | SPI data out (SPI5) |
| DCLK | PF7 | SPI clock (SPI5) |
| PENIRQ | PB0 | Touch interrupt |

**Note**: The touchscreen overlay connects to X-, Y-, X+, Y+ pins on the XPT2046.

## Software Setup

### Enable the Driver

In your CMakeLists.txt, enable the XPT2046 driver:

```cmake
option(USE_XPT2046 "Include XPT2046 resistive touchscreen" ON)
```

Or enable it during cmake configuration:
```bash
cmake -DUSE_XPT2046=ON ..
```

## Basic Usage

### Include Headers

```c
#include "xpt2046.h"
// SPI is handled by the central SPI driver
```

### Initialize the Touchscreen

```c
// Declare touchscreen handle
XPT2046_Handle_t hxpt;

// Initialize with SPI5, pins PB15(CS), PB0(IRQ), 320x480 display
XPT2046_StatusTypeDef status = XPT2046_Init(&hxpt,
                                          GPIOB, GPIO_PIN_15,  // CS
                                          GPIOB, GPIO_PIN_0,    // IRQ
                                          320, 480);            // Display size

if (status != XPT2046_OK) {
    // Handle initialization error
}
```

### Basic Touch Detection

#### Check if Touched
```c
if (XPT2046_IsTouched(&hxpt)) {
    // Touchscreen is being touched
    printf("Touch detected!\n");
} else {
    // No touch
    printf("No touch\n");
}
```

#### Read Touch Coordinates
```c
XPT2046_TouchPoint_t touch;

if (XPT2046_ReadTouch(&hxpt, &touch) == XPT2046_OK) {
    printf("Touch at: X=%d, Y=%d, Pressure=%d\n",
           touch.x, touch.y, touch.pressure);
} else {
    printf("No touch detected\n");
}
```

### Touch State Tracking

For applications that need to track touch state changes:

```c
// Update touch state (call this regularly, e.g., in main loop)
XPT2046_Update(&hxpt);

// Get current touch state
XPT2046_TouchPoint_t current_touch;
XPT2046_GetTouch(&hxpt, &current_touch);

// Check touch state
switch (current_touch.state) {
    case XPT2046_STATE_RELEASED:
        printf("Touch released\n");
        break;
    case XPT2046_STATE_PRESSED:
        printf("Touch pressed at %d,%d\n", current_touch.x, current_touch.y);
        break;
    case XPT2046_STATE_HELD:
        printf("Touch held at %d,%d\n", current_touch.x, current_touch.y);
        break;
}
```

## Touchscreen Calibration

### Basic Coordinate Mapping

The driver automatically maps raw ADC values to display coordinates, but you may need calibration for accuracy.

```c
// Set custom calibration matrix (7 values)
// Format: {x_scale, x_offset, xy_coeff, y_scale, y_offset, yx_coeff, divisor}
uint16_t calibration_matrix[7] = {1, 0, 0, 1, 0, 0, 1};
XPT2046_SetCalibration(&hxpt, calibration_matrix);
```

### Coordinate Flipping

If your touchscreen coordinates are inverted:

```c
// Configure coordinate flipping
XPT2046_Config_t config = {
    .hspi = &hspi1,
    .cs_port = GPIOB,
    .cs_pin = GPIO_PIN_15,
    .irq_port = GPIOB,
    .irq_pin = GPIO_PIN_1,
    .width = 320,
    .height = 480,
    .flip_x = true,   // Flip X coordinates
    .flip_y = false,  // Don't flip Y coordinates
};

XPT2046_Config(&hxpt, &config);
```

## Complete Example

### Simple Touch Demo

```c
#include "xpt2046.h"
#include "spi.h"
#include "usart.h"  // For printf

XPT2046_Handle_t hxpt;

void XPT2046_Demo(void) {
    // Initialize touchscreen
    XPT2046_Init(&hxpt, &hspi1, GPIOB, GPIO_PIN_15, GPIOB, GPIO_PIN_1, 320, 480);
    
    printf("XPT2046 Touchscreen Demo\n");
    
    while (1) {
        // Update touch state
        XPT2046_Update(&hxpt);
        
        // Get current touch
        XPT2046_TouchPoint_t touch;
        XPT2046_GetTouch(&hxpt, &touch);
        
        // Display touch information
        if (touch.state == XPT2046_STATE_PRESSED) {
            printf("Touch pressed at: X=%d, Y=%d, Pressure=%d\n",
                   touch.x, touch.y, touch.pressure);
        } else if (touch.state == XPT2046_STATE_HELD) {
            printf("Touch held at: X=%d, Y=%d\n", touch.x, touch.y);
        } else if (touch.state == XPT2046_STATE_RELEASED) {
            printf("Touch released\n");
        }
        
        // Small delay
        HAL_Delay(50);
    }
}
```

### Combined with ILI9488 Display

```c
#include "ili9488.h"
#include "xpt2046.h"

ILI9488_Handle_t hili;
XPT2046_Handle_t hxpt;

void Touchscreen_Paint_Demo(void) {
    // Initialize display
    ILI9488_Init(&hili, &hspi1, GPIOB, GPIO_PIN_12, GPIOB, GPIO_PIN_13, GPIOB, GPIO_PIN_14);
    ILI9488_Clear(&hili, ILI9488_COLOR_WHITE);
    
    // Initialize touchscreen
    XPT2046_Init(&hxpt, &hspi1, GPIOB, GPIO_PIN_15, GPIOB, GPIO_PIN_1, 320, 480);
    
    printf("Touch to draw!\n");
    
    while (1) {
        XPT2046_Update(&hxpt);
        
        XPT2046_TouchPoint_t touch;
        XPT2046_GetTouch(&hxpt, &touch);
        
        if (touch.state == XPT2046_STATE_PRESSED || touch.state == XPT2046_STATE_HELD) {
            // Draw a dot at touch position
            ILI9488_DrawFilledCircle(&hili, touch.x, touch.y, 3, ILI9488_COLOR_RED);
        }
        
        HAL_Delay(10);
    }
}
```

## Touch States

The driver tracks three touch states:

- **`XPT2046_STATE_RELEASED`**: No touch detected
- **`XPT2046_STATE_PRESSED`**: New touch just detected
- **`XPT2046_STATE_HELD`**: Touch is being held/continued

Use these states to implement different behaviors for touch down vs. touch drag operations.

## Pressure Detection

The XPT2046 provides pressure information:

```c
XPT2046_TouchPoint_t touch;
XPT2046_ReadTouch(&hxpt, &touch);

if (touch.pressure > 100) {  // Light touch
    // Handle light touch
} else if (touch.pressure > 500) {  // Firm touch
    // Handle firm touch
}
```

Pressure values range from 0 (no touch) to ~1000+ (firm press).

## Performance Considerations

1. **SPI Sharing**: If sharing SPI with display, ensure proper chip select management
2. **Interrupt Usage**: Use the PENIRQ pin for efficient touch detection
3. **Debouncing**: The driver includes basic debouncing, but you may need additional filtering
4. **Calibration**: Always calibrate for accurate touch coordinates
5. **Update Frequency**: Call `XPT2046_Update()` at appropriate intervals (10-50ms)

## Troubleshooting

### No Touch Detection
- Check PENIRQ pin connection and pull-up resistor
- Verify SPI connections (MOSI, MISO, SCK, CS)
- Ensure proper power supply (3.3V)
- Check touchscreen overlay connections

### Inaccurate Coordinates
- Perform touchscreen calibration
- Check coordinate flipping settings
- Verify display dimensions match touchscreen
- Check for electromagnetic interference

### Erratic Touch Behavior
- Add debounce delay in touch polling
- Check for SPI bus conflicts
- Verify interrupt pin is not floating
- Consider adding noise filtering

## Advanced Features

### Custom Configuration

```c
XPT2046_Config_t config = {
    .hspi = &hspi1,
    .cs_port = GPIOB,
    .cs_pin = GPIO_PIN_15,
    .irq_port = GPIOB,
    .irq_pin = GPIO_PIN_1,
    .width = 320,
    .height = 480,
    .flip_x = false,
    .flip_y = false,
    // Custom calibration matrix
    .calibration = {1, 0, 0, 1, 0, 0, 1}
};

XPT2046_Config(&hxpt, &config);
```